# Quasi-Frozen Orbit and ELFO Orbit Elements Analysis
This notebook performs long term orbit propagation for Quasi-frozen LLO and ELFO

## Choose Orbit and Setup Options
Elliptical Lunar Frozen Orbit and Low Lunar Orbit available

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib
import plotly.graph_objects as go 
import matplotlib.pyplot as plt

import pylupnt as pnt

from qllo import qllo_opt

In [ ]:
# orbit = 'LLO'
orbit = 'ELFO'

# time
t0 = pnt.gregorian2time(2022, 1, 2, 0, 0, 0)

if orbit == 'LLO':
    print("LLO")

    a_init = 1850.0
    e_init = 0.05

    coe0 = qllo_opt(a0=a_init, e0=e_init)
    rv0_op = pnt.classical2cart(coe0, pnt.GM_MOON)
    rv0_mi = pnt.convert_frame(t0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)
    
elif orbit == 'ELFO':
    print("ELFO")
    # Classical Orbital Elements (COE)
    a = 6541.4  # [km] Semi-major axis
    e = 0.6000  # [--] Eccentricity
    i = 56.2 * pnt.RAD  # [deg] Inclination
    O = 0.00 * pnt.RAD  # [deg] Right ascension of the ascending node
    w = 90.0 * pnt.RAD  # [deg] Argument of perigee
    M = 0.00 * pnt.RAD  # [deg] Mean anomaly

    coe0 = np.array([a, e, i, O, w, M])   # in op frame
    rv0_op = pnt.classical2cart(coe0, pnt.GM_MOON)
    rv0_mi = pnt.convert_frame(t0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)

coe_mi = pnt.cart2classical(rv0_mi, pnt.GM_MOON)
T_orbit = 2 * np.pi * np.sqrt(coe_mi[0]**3 / pnt.GM_MOON)

dt_step = 60.0 * pnt.SECS_MINUTE
dt_prop = 1.0 * pnt.SECS_MINUTE
t_end = 50 * pnt.SECS_DAY

# print the orbital elements
print("Initial state (OP Frame):")
print(" a [km]:", coe0[0])
print(" e [-]:", coe0[1])
print(" i [deg]:", coe0[2] * pnt.DEG)
print(" O [deg]:", coe0[3] * pnt.DEG)
print(" w [deg]:", coe0[4] * pnt.DEG)
print(" M [deg]:", coe0[5] * pnt.DEG)
print(" ")
print("Initial state (CI Frame):")
print("a [km]:", coe_mi[0])
print("e [-]:", coe_mi[1])
print("i [deg]:", coe_mi[2] * pnt.DEG)
print("O [deg]:", coe_mi[3] * pnt.DEG)
print("w [deg]:", coe_mi[4] * pnt.DEG)
print("M [deg]:", coe_mi[5] * pnt.DEG)


## Propagate Orbit using Multi-body Dynamics
The list of perturbations are as follows
- 50x50 Moon gravity
- Third body: Earth, Sun

For integration, we use the RKF-45 integrator (adaptive stepsize) with RelTol=1e-12, AbsTol=1e-12

In [ ]:
dyn = pnt.NBodyDynamics(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-12, reltol=1e-12))

dyn.add_body(pnt.Body.Moon(50, 50))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_time_step(dt_prop)
dyn.set_frame(pnt.MOON_CI)

In [ ]:
tspan = t0 + np.arange(0, t_end, dt_step)

rv_prop_mi = dyn.propagate(rv0_mi, t0, tspan, progress=True)  # in Moon Inertial frame

In [ ]:
# visualize orbit
rv_prop_pa = pnt.convert_frame(t0 * np.ones(np.size(tspan)), rv_prop_mi, pnt.MOON_CI, pnt.MOON_OP, rotate_only=True)  # in Moon Principal Axes frame
orb_plot = np.zeros((2, len(tspan), 6))
orb_plot[0, :, :] = rv_prop_mi
orb_plot[1, :, :] = rv_prop_pa

fig = go.Figure()
pnt.plot.plot_body(fig, pnt.MOON)
pnt.plot.plot_orbits(
    fig,
    orb_plot,
    color = ['blue', 'red'],
)
pnt.plot.set_view(fig, azimuth=-130, elevation=10, zoom=2.5)
fig.update_layout(width=400, height=400)
# set labels (mci, PA)
text_fig = """
<b>Satellite orbit</b><br>
Red: Moon OP<br>
Blue: Moon CI<br>
"""

fig.add_annotation(
    text=text_fig, **dict(x=0.01, y=0.98, align = 'left', xanchor = 'left', showarrow=False)
)
fig.show()

In [ ]:
# Plot
coe_case1_op = pnt.cart2classical(rv_prop_pa, pnt.GM_MOON)
labels = [
    "Semi-major axis [km]",
    "Eccentricity [-]",
    "Inclination [deg]",
    "Right Asc. [deg]",
    "Arg. of Periapsis [deg]",
    "Mean Anomaly [deg]",
]

fig = plt.figure(figsize=(8, 6))
matplotlib.rcParams.update({"font.size": 12})
x = (tspan - t0) / pnt.SECS_DAY
if orbit == 'LLO':
    plt.suptitle("Orbital Elements in OP frame (Quasi-Frozen LLO)")
elif orbit == 'ELFO':
    plt.suptitle("Orbital Elements in OP frame (ELFO)")
    
for i in range(6):
    plt.subplot(3, 2, i + 1)
    y = coe_case1_op[:, i] if i < 2 else coe_case1_op[:, i] * pnt.DEG
    plt.plot(x, y)
    plt.xlabel("Days past " + pnt.time2gregorian_string(t0) + " TAI")
    plt.ylabel(labels[i])
    plt.grid()
    if i < 5:
        plt.xlim(x[0], x[-1])
    else:
        plt.xlim(x[0], x[100])
plt.tight_layout()
plt.show()

## Lunar Orientation Angle
Another parameter to be fitted within the framework is the orientation angle of the lunar surface. We extract these parameters from the SPICE kernel file and see how it changes over 50 days


In [ ]:
# Extract the lunar orientation angles
angles = np.zeros((len(tspan), 6))
for i, t in enumerate(tspan):
    angles[i, :] = pnt.get_lunar_orientation_angles(t)

# Plot
# phi, theta, psi, phi_dot, theta_dot, psi_dot
labels = [
    "Phi [deg]",
    "Theta [deg]",
    "Psi [deg]",
    "Phi_dot [deg/s]",
    "Theta_dot [deg/s]",
    "Psi_dot [deg/s]",
]

fig = plt.figure(figsize=(12, 6))
matplotlib.rcParams.update({"font.size": 12})
x = (tspan - t0) / pnt.SECS_DAY
plt.suptitle("Lunar Orientation Angles")
for i in range(6):
    plt.subplot(3, 2, i + 1)
    y = angles[:, i] * pnt.DEG
    if i <= 2:
        y = np.mod(y, 360)
    plt.plot(x, y)
    plt.xlabel("Days past " + pnt.time2gregorian_string(t0) + " TAI")
    plt.ylabel(labels[i])
    plt.grid()
    plt.xlim(x[0], x[-1])

plt.tight_layout()
plt.show()